In [1]:
import pandas as pd
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [7]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

feature_columns = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip', 'City', 'Borough',
                    'Open Data Channel Type', 'Latitude', 'Longitude',
                    'hour_of_day', 'day_of_week', 'month',
                    'is_weekend', 'open_requests_past_24h']

X = df[feature_columns]
y = df['resolution_class_int']

#Segregate columsn for preprocessing
categorical_cols = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip', 'City', 'Borough',
                    'Open Data Channel Type', 'hour_of_day','day_of_week', 'month',
                    'is_weekend']
numeric_cols = ['Latitude','Longitude','open_requests_past_24h']

# Preprocessing -> Normalization + encoding categories
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'  # numeric features untouched
)

xg = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",            
    enable_categorical=True,
    n_jobs=8,
    verbosity=3,
    random_state=42      
)

# 4. Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', xg)
])

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')

print(scores)

# 6. Fit model
model.fit(X_train, y_train)

# 7. Predict
y_pred = model.predict(X_test)

[21:26:32] ======== Monitor (0): HostSketchContainer ========
[21:26:32] AllReduce: 0.002994s, 1 calls @ 2994us

[21:26:32] MakeCuts: 0.003019s, 1 calls @ 3019us

[21:26:32] INFO: /workspace/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1272686, 15, 19090290).
[21:26:32] DEBUG: /workspace/src/gbm/gbtree.cc:127: Using tree method: 3
[21:26:53] ======== Monitor (0): GBTree ========
[21:26:53] BoostNewTrees: 19.3943s, 100 calls @ 19394270us

[21:26:53] CommitModel: 0.000145s, 100 calls @ 145us

[21:26:53] ======== Monitor (0): HistUpdater ========
[21:26:53] BuildHistogram: 6.85992s, 4200 calls @ 6859921us

[21:26:53] EvaluateSplits: 0.299307s, 4800 calls @ 299307us

[21:26:53] InitData: 0.381278s, 600 calls @ 381278us

[21:26:53] InitRoot: 2.42416s, 600 calls @ 2424162us

[21:26:53] LeafPartition: 0.000197s, 600 calls @ 197us

[21:26:53] UpdatePosition: 5.58297s, 4800 calls @ 5582967us

[21:26:53] UpdatePredictionCache: 1.74684s, 600 calls @ 1746839us



In [9]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# print(scores)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7982432981741532

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93    331345
           1       0.54      0.71      0.61     76042
           2       0.37      0.28      0.31     27143
           3       0.46      0.14      0.21     17086
           4       0.35      0.18      0.24     15160
           5       0.52      0.32      0.39     10482

    accuracy                           0.80    477258
   macro avg       0.53      0.43      0.45    477258
weighted avg       0.78      0.80      0.78    477258


Confusion Matrix:
 [[311345  17621   1299    121    324    635]
 [ 16433  53730   4035    437    958    449]
 [  4519  12647   7491    955   1095    436]
 [  2514   6951   3345   2354   1431    491]
 [  2236   5370   2832    907   2733   1082]
 [  1467   2751   1448    290   1211   3315]]


In [10]:
import numpy as np
np.save("../results/xgboost_predictions_100.npy", y_pred)

In [8]:
import numpy as np
y_pred = np.load("../results/random_forest_predictions_150.npy")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt 

sns.heatmap(confusion_matrix(y_test, y_pred), annot=False)
plt.savefig("../results/cm_random_forest.png")

In [15]:
tes = df[categorical_cols]
tes.describe(include='all')

,Agency,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,City,Borough,Open Data Channel Type,is_weekend
count,2386288,2386288,2386288,2386288,2386288,2386288,2386288,2386288,2386288,2.386288e+06
unique,15,168,852,137,243,467120,51,6,5,NaN
top,NYPD,ILLEGAL PARKING,LOUD MUSIC/PARTY,STREET/SIDEWALK,10466,UNKNOWN,BROOKLYN,BROOKLYN,ONLINE,NaN
freq,1277321,435694,355810,870943,82021,77782,720072,720073,1041275,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.765500e-01
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.472921e-01
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00
